# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [2]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

⚠️ API Key not found! Please check your .env file.


## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints. Depending on their exact documentation, the API key might need to be passed as a query parameter (e.g. `?apikey=...`) or a specific header.

In [3]:
# The base URL typically looks something like this. 
# You may need to tweak this based on the exact footballdata.io docs.
BASE_URL = "https://api.footballdata.io/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    # Standard Authorization approaches. You might need to change this 
    # to `"X-API-Key": API_KEY` or include it in `params` depending on the docs.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Test Connection
Let's do a basic test to make sure the key works and we can view the data payload structure.

In [4]:
# Try fetching a list of active leagues or today's matches to test the connection
test_endpoint = "leagues"

print(f"Fetching endpoint: /{test_endpoint}...")
data = fetch_footballdata(test_endpoint)

if data:
    print("\n✅ Success! Here is a snippet of the response:\n")
    print(json.dumps(data, indent=2)[:1000] + "\n...[truncated]")
else:
    print("\n⚠️ Failed to retrieve data. Check endpoint path or auth headers.")

Fetching endpoint: /leagues...
❌ Error fetching data: HTTPSConnectionPool(host='api.footballdata.io', port=443): Max retries exceeded with url: /v1/leagues (Caused by NameResolutionError("HTTPSConnection(host='api.footballdata.io', port=443): Failed to resolve 'api.footballdata.io' ([Errno 11001] getaddrinfo failed)"))

⚠️ Failed to retrieve data. Check endpoint path or auth headers.
